In [1]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [4]:
# data = pd.read_stata(r"datos/ECU_2004m12_BID.dta", convert_categoricals=False) # para bases de stata

data3 = pd.read_stata(r"Z:\survey\ECU\ENEMDU\2005\m3\data_orig\ecu05-mar.dta", convert_categoricals=False) # para bases de stata
data6 = pd.read_stata(r"Z:\survey\ECU\ENEMDU\2005\m6\data_orig\ecu05-jun.dta", convert_categoricals=False) # para bases de stata
data9 = pd.read_stata(r"Z:\survey\ECU\ENEMDU\2005\m9\data_orig\ecu05-sep.dta", convert_categoricals=False) # para bases de stata
data12 = pd.read_stata(r"Z:\survey\ECU\ENEMDU\2005\m12\data_orig\per12_2005.dta", convert_categoricals=False) # para bases de stata

## Revisar los datos

| marzo | junio | septiembre | diciembre |
|-----------|-----------|-----------|-----------|
| area  | area  | area  | area  |
| ciudad  | ciudad  | ciudad  | ciudad  |
| zona  | zona  | zona  | zona  |
| sector  | sector  | sector  | sector  |
| panelm  | panelm  | panelm  | panelm  |
| vivienda  | vivienda  | vivienda  | vivienda  |
| hogar  | hogar  | hogar  | hogar  |
| sexo  | sexo  | sexo  | sexo  |
| edad  | edad  | edad  | edad  |
| pe63  | pe63  | pe63  | pe63  |
| fexp  | fexp  | fexp  | fexp  |
| trabajo  | trabajo  | trabajo  | trabajo  |

En esta encuesta tenemos separadas cuatro diferentes bases para cada trimestre, esto cambia la lógica que habíamos tenido hasta ahora así que de aquí en adelante cambiamos algo del código, mantenemos de acuerdo a las etiquetas de las variables pe63 como la variable de ingreso laboral monetario de la actividad principal asalariada para mantener la concordancia con el resto de los años.

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no trabajando, estas variables las usamos antes para identificar la condición de trabajo para diferentes meses en encuestas anuales o incompletas donde asumíamos que mantenía el mismo salario si estaba ocupado en ese mes, sin mbargo estas variables tenían el problema de no corresponder de forma exacta con el año o mes de la encuesta. Ahora sin embargo podemos cambiar las suposiciones y solamente asumir que si la variable 'trabajando' que pregunta si el individuo trabajó la semana pasada se cumple vamos a asumir que trabajo durante todo el trimestre, de esta manera podemos mejorar las suposiciones de ocupación mensual, mantenemos la idea de que si el individuo trabajo recibe su ingreso laboral reportado.

In [5]:
columnas = pd.Index(['area', 'ciudad', 'zona', 'sector', 'panelm',
            'vivienda', 'hogar', 'sexo', 'edad', 'pe63', 'trabajo',
            'fexp'])

Filtramos solo las columnas de interés para alivar el peso en la memoria

In [6]:
data3 = data3[columnas]
data6 = data6[columnas]
data9 = data9[columnas.intersection(data9.columns)]
data12 = data12[columnas.intersection(data12.columns)]

Creamos una variable de ingreso laboral que es igual al ingreso por asalariado y limpiamos según los valores de ingrl, para mantener ambas variables para cada base consistente

In [11]:
data3['pe63'] = pd.to_numeric(data3['pe63'], errors='coerce')
data3['pe63'] = data3['pe63'].apply(lambda x: np.nan if x > 4840 else x)

data6['pe63'] = pd.to_numeric(data6['pe63'], errors='coerce')
data6['pe63'] = data6['pe63'].apply(lambda x: np.nan if x > 7793 else x)

data9['pe63'] = pd.to_numeric(data9['pe63'], errors='coerce')
data9['pe63'] = data9['pe63'].apply(lambda x: np.nan if x > 8000 else x)

data12['pe63'] = pd.to_numeric(data12['pe63'], errors='coerce')
data12['pe63'] = data12['pe63'].apply(lambda x: np.nan if x > 8000 else x)

In [12]:
data3['ingr'] = data3['pe63']
data6['ingr'] = data6['pe63']
data9['ingr'] = data9['pe63']
data12['ingr'] = data12['pe63']

Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingr' siempre que reportan estar ocupados la semana pasada, de acuerdo a la variable 'trabajo'

In [22]:
data3['ingr_t1'] = data3.apply(lambda x: x['ingr'] if x['trabajo'] == 1 else np.nan, axis=1)

data6['ingr_t2'] = data6.apply(lambda x: x['ingr'] if x['trabajo'] == 1 else np.nan, axis=1)

data9['ingr_t3'] = data9.apply(lambda x: x['ingr'] if x['trabajo'] == 1 else np.nan, axis=1)

data12['ingr_t4'] = data12.apply(lambda x: x['ingr'] if x['trabajo'] == 1 else np.nan, axis=1)

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014

In [23]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 2005]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc

In [30]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Sierra': fila['Sierra'],
        'Costa': fila['Costa'],
        'Guayaquil': fila['Guayaquil'],
        'Esmeraldas': fila['Esmeraldas'],
        'Machala': fila['Machala'],
        'Manta': fila['Manta'],
        'Quito': fila['Quito'],
        'Loja': fila['Loja'],
        'Cuenca': fila['Cuenca'],
        'Ambato': fila['Ambato']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Sierra': fila['Sierra'],
        'Costa': fila['Costa'],
        'Guayaquil': fila['Guayaquil'],
        'Esmeraldas': fila['Esmeraldas'],
        'Machala': fila['Machala'],
        'Manta': fila['Manta'],
        'Quito': fila['Quito'],
        'Loja': fila['Loja'],
        'Cuenca': fila['Cuenca'],
        'Ambato': fila['Ambato']
    }
     for _, fila in datos_base.iterrows()
     }

### Creamos identificadores para las ciudades siguiendo los códigos del INEC y para los trimestres

In [34]:
# Corregimos los códigos para usarlos cómo texto
data3['ciudad'] = data3['ciudad'].apply(str)
data3['ciudad'] = data3['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)
data3['ciudad_2'] = data3['ciudad'].apply(lambda x: x[:4])

data6['ciudad'] = data6['ciudad'].apply(str)
data6['ciudad'] = data6['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)
data6['ciudad_2'] = data6['ciudad'].apply(lambda x: x[:4])

data9['ciudad'] = data9['ciudad'].apply(str)
data9['ciudad'] = data9['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)
data9['ciudad_2'] = data9['ciudad'].apply(lambda x: x[:4])

data12['ciudad'] = data12['ciudad'].apply(str)
data12['ciudad'] = data12['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)
data12['ciudad_2'] = data12['ciudad'].apply(lambda x: x[:4])

Diccionario ciudades disponibles

In [39]:
parroquia_dict = {
    '0101': 'Cuenca',
    '0901': 'Guayaquil',
    '0801': 'Esmeraldas',
    '0701': 'Machala',
    '1308': 'Manta',
    '1701': 'Quito',
    '1101': 'Loja',
    '1801': 'Ambato'
}

def get_parroquia(codigo):
    if codigo in parroquia_dict:
        return parroquia_dict[codigo]
    elif codigo[:2] in ['01', '02', '03', '04', '05', '06', '10', '11', '17', '18']:
        return 'Sierra'
    elif codigo[:2] in ['07', '08', '09', '12', '13', '23', '24']:
        return 'Costa'
    else:
        return 'Nacional'

data3['ciudad_asignada'] = data3['ciudad_2'].apply(get_parroquia)
data6['ciudad_asignada'] = data6['ciudad_2'].apply(get_parroquia)
data9['ciudad_asignada'] = data9['ciudad_2'].apply(get_parroquia)
data12['ciudad_asignada'] = data12['ciudad_2'].apply(get_parroquia)

In [48]:
data3['ciudad_asignada'].value_counts()

ciudad_asignada
Costa         5018
Guayaquil     4612
Quito         3827
Sierra        3160
Nacional      2179
Machala       1968
Cuenca        1942
Ambato         442
Manta          438
Esmeraldas     390
Loja           309
Name: count, dtype: int64

### Asignamos el ipc correspondiente según ciudad correspondiente

$\begin{equation}
    ingr_{USD-base-2014}^{i} = ingr_{dólares}^{i}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Desde 2000 en adelante ya no es necesario utilizar el tipo de cambio debido al cambio de moneda

In [45]:
# Función que asigna valores correspondientes
def asigna_ipc(fila, trimestre):
    return ipc_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila, trimestre):
    return ipc_base_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

In [46]:
data3['ipc_t1'] = data3.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
data3['ipc_base_t1'] = data3.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)

data6['ipc_t2'] = data6.apply(lambda fila: asigna_ipc(fila, 2), axis=1)
data6['ipc_base_t2'] = data6.apply(lambda fila: asigna_ipc_base(fila, 2), axis=1)

data9['ipc_t3'] = data9.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data9['ipc_base_t3'] = data9.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)

data12['ipc_t4'] = data12.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data12['ipc_base_t4'] = data12.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)

In [49]:
# Calculamos el deflactor
data3['def_t1'] = (data3['ipc_base_t1'] / data3['ipc_t1'])
data6['def_t2'] = (data6['ipc_base_t2'] / data6['ipc_t2'])
data9['def_t3'] = (data9['ipc_base_t3'] / data9['ipc_t3'])
data12['def_t4'] = (data12['ipc_base_t4'] / data12['ipc_t4'])

Ingreso promedio en el trimeste

In [50]:
data3['ingr_t1_r'] = data3['ingr_t1'] * data3['def_t1']
data6['ingr_t2_r'] = data6['ingr_t2'] * data6['def_t2']
data9['ingr_t3_r'] = data9['ingr_t3'] * data9['def_t3']
data12['ingr_t4_r'] = data12['ingr_t4'] * data12['def_t4']

In [51]:
print(data3['ingr_t1_r'].mean())
print(data6['ingr_t2_r'].mean())
print(data9['ingr_t3_r'].mean())
print(data12['ingr_t4_r'].mean())

356.67085277178626
354.8664526528096
360.1138207823786
285.3462136256262


## Calculo ingreso de los hogares

In [52]:
columnas_idef = pd.Index(['area', 'ciudad', 'zona', 'sector', 'vivienda',
       'hogar'])

data3['idef_hogar'] = data3[columnas_idef].astype(str).agg(''.join, axis=1)
data6['idef_hogar'] = data6[columnas_idef].astype(str).agg(''.join, axis=1)
data9['idef_hogar'] = data9[
    columnas_idef.intersection(data9.columns)
    ].astype(str).agg(''.join, axis=1)
data12['idef_hogar'] = data12[
    columnas_idef.intersection(data12.columns)
    ].astype(str).agg(''.join, axis=1)

print(len(data3['idef_hogar'].unique()))
print(len(data6['idef_hogar'].unique()))
print(len(data9['idef_hogar'].unique()))
print(len(data12['idef_hogar'].unique()))

1533
1544
3020
5014


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [53]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [54]:
data3['ingr_t1_h'] = data3.groupby('idef_hogar')['ingr_t1_r'].transform(sum_with_na)
data6['ingr_t2_h'] = data6.groupby('idef_hogar')['ingr_t2_r'].transform(sum_with_na)
data9['ingr_t3_h'] = data9.groupby('idef_hogar')['ingr_t3_r'].transform(sum_with_na)
data12['ingr_t4_h'] = data12.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)

In [55]:
print(data3['ingr_t1_h'].mean())
print(data6['ingr_t2_h'].mean())
print(data9['ingr_t3_h'].mean())
print(data12['ingr_t4_h'].mean())

1432.4398287452461
1445.0942911711566
849.8717131770483
1088.8552149930904


## Sacamos edades negativas y mayores a 100 años

In [56]:
print(len(data3))
print(len(data6))
print(len(data9))
print(len(data12))

24285
24424
24105
77050


Transformamos las variables de edad a numericas para evitar problemas

In [57]:
data3['edad'] = pd.to_numeric(data3['edad'], errors='coerce')
data6['edad'] = pd.to_numeric(data6['edad'], errors='coerce')
data9['edad'] = pd.to_numeric(data9['edad'], errors='coerce')
data12['edad'] = pd.to_numeric(data12['edad'], errors='coerce')

In [58]:
data3 = data3.loc[(data3['edad'] >= 0) & (data3['edad'] < 100)]
data6 = data6.loc[(data6['edad'] >= 0) & (data6['edad'] < 100)]
data9 = data9.loc[(data9['edad'] >= 0) & (data9['edad'] < 100)]
data12 = data12.loc[(data12['edad'] >= 0) & (data12['edad'] < 100)]

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [59]:
k = 0.4
s = 0.9

In [60]:
# Si es necesario calcular el número de niños
data3['es_nino'] = data3['edad'] < 10
data3['ninos'] = data3.groupby('idef_hogar')['es_nino'].transform('sum')

data6['es_nino'] = data6['edad'] < 10
data6['ninos'] = data6.groupby('idef_hogar')['es_nino'].transform('sum')

data9['es_nino'] = data9['edad'] < 10
data9['ninos'] = data9.groupby('idef_hogar')['es_nino'].transform('sum')

data12['es_nino'] = data12['edad'] < 10
data12['ninos'] = data12.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data3['es_adulto'] = data3['edad'] > 10
data3['adultos'] = data3.groupby('idef_hogar')['es_adulto'].transform('sum')

data6['es_adulto'] = data6['edad'] > 10
data6['adultos'] = data6.groupby('idef_hogar')['es_adulto'].transform('sum')

data9['es_adulto'] = data9['edad'] > 10
data9['adultos'] = data9.groupby('idef_hogar')['es_adulto'].transform('sum')

data12['es_adulto'] = data12['edad'] > 10
data12['adultos'] = data12.groupby('idef_hogar')['es_adulto'].transform('sum')

In [61]:
data3['escala'] = (data3['adultos'] + k * data3['ninos']) ** s
data6['escala'] = (data6['adultos'] + k * data6['ninos']) ** s
data9['escala'] = (data9['adultos'] + k * data9['ninos']) ** s
data12['escala'] = (data12['adultos'] + k * data12['ninos']) ** s

In [62]:
data3['ingr_t_t1'] = data3['ingr_t1_h'] / data3['escala']
data6['ingr_t_t2'] = data6['ingr_t2_h'] / data6['escala']
data9['ingr_t_t3'] = data9['ingr_t3_h'] / data9['escala']
data12['ingr_t_t4'] = data12['ingr_t4_h'] / data12['escala']

In [63]:
print(data3['ingr_t_t1'].mean())
print(data6['ingr_t_t2'].mean())
print(data9['ingr_t_t3'].mean())
print(data12['ingr_t_t4'].mean())

131.19001097515533
132.3818906555686
139.75648930392853
100.89694970574483


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [64]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

In [65]:
datos_final = pd.DataFrame(index=['t1', 't2', 't3', 't4'], columns=['fgt0', 'fgt1', 'fgt2', 'a25', 'a50', 'a75', 'ingreso_promedio'])

In [66]:
data3['persona_fexp'] = 1 * data3['fexp']
data6['persona_fexp'] = 1 * data6['fexp']
data9['persona_fexp'] = 1 * data9['fexp']
data12['persona_fexp'] = 1 * data12['fexp']

In [67]:
dict_t = {1:data3, 2:data6, 3:data9, 4:data12}

In [68]:
for t in [1, 2, 3, 4]:
    col_ingr = f'ingr_t_t{t}'
    col_pobres = f'pobres_t{t}'

    # una columna que identifica a quienes están por debajo de la línea de pobreza por trimestre
    dict_t[t][col_pobres] = (
        (dict_t[t][col_ingr] - umbral_dict.get(t)) < 0
    ).astype(int)

In [69]:
print("pobreza t1: ", (data3['pobres_t1'] * data3['fexp']).sum()/data3.loc[data3['ingr_t_t1'] >= 0]['persona_fexp'].sum())
print("pobreza t2: ", (data6['pobres_t2'] * data6['fexp']).sum()/data6.loc[data6['ingr_t_t2'] >= 0]['persona_fexp'].sum())
print("pobreza t3: ", (data9['pobres_t3'] * data9['fexp']).sum()/data9.loc[data9['ingr_t_t3'] >= 0]['persona_fexp'].sum())
print("pobreza t4: ", (data12['pobres_t4'] * data12['fexp']).sum()/data12.loc[data12['ingr_t_t4'] >= 0]['persona_fexp'].sum())

pobreza t1:  0.25763025351308777
pobreza t2:  0.2645983797605204
pobreza t3:  0.29742039844281243
pobreza t4:  0.35897298876923034


In [70]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = dict_t[t].loc[dict_t[t][f'ingr_t_t{t}'] >= 0].copy()

    # Calculamos una columna de pobres
    df_temp['pobres'] = (df_temp[f'ingr_t_t{t}'] - umbral_dict[t]) < 0

    # Ratio de pobres sobre el total
    ratio = (umbral_dict[t] - df_temp[f'ingr_t_t{t}']) / umbral_dict[t]

    # Calculamos el índice para alpha 0, 1 y 2 solo donde 'pobres' == True.
    for i in range(3):
        col = f'fgt{i}'
        df_temp[col] = np.where(df_temp['pobres'], ratio**i, 0)

    # Cálculo del índice ponderado: se usa el factor de expansión como peso
    peso_total = df_temp['fexp'].sum()
    fgt0 = (df_temp['fgt0'] * df_temp['fexp']).sum() / peso_total
    fgt1 = (df_temp['fgt1'] * df_temp['fexp']).sum() / peso_total
    fgt2 = (df_temp['fgt2'] * df_temp['fexp']).sum() / peso_total
    
    # Guardamos los resultados
    datos_final.loc[f't{t}', 'fgt0'] = fgt0
    datos_final.loc[f't{t}', 'fgt1'] = fgt1
    datos_final.loc[f't{t}', 'fgt2'] = fgt2

In [71]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.25763,0.107017,0.062802,NaN,NaN,NaN,NaN
t2,0.264598,0.099451,0.052628,NaN,NaN,NaN,NaN
t3,0.29742,0.122079,0.070364,NaN,NaN,NaN,NaN
t4,0.358973,0.156552,0.093671,NaN,NaN,NaN,NaN


## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [72]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = dict_t[t].loc[dict_t[t][f'ingr_t_t{t}'] >= 0].copy()

    # Suma total de los factores de expansión para el trimestre
    peso_total = df_temp['fexp'].sum()
    
    # Ingreso promedio ponderado
    mu = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total

    # Calculamos el índice A para epsilon 0.25, 0.5 y 0.75 utilizando los pesos
    indices = {}
    for i in [0.25, 0.5, 0.75]:
        A_i = ((df_temp[f'ingr_t_t{t}']**(1-i) * df_temp['fexp']).sum() / peso_total)**(1/(1-i))
        indices[i] = A_i

    # Ratio de pobreza con el índice total (aplicando la fórmula)
    a25 = 1 - 1/mu * indices[0.25]
    a50 = 1 - 1/mu * indices[0.5]
    a75 = 1 - 1/mu * indices[0.75]

    # Guardamos los resultados
    datos_final.loc[f't{t}', 'a25'] = a25
    datos_final.loc[f't{t}', 'a50'] = a50
    datos_final.loc[f't{t}', 'a75'] = a75


In [73]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.25763,0.107017,0.062802,0.077345,0.151542,0.226253,NaN
t2,0.264598,0.099451,0.052628,0.082729,0.158865,0.232562,NaN
t3,0.29742,0.122079,0.070364,0.097421,0.186945,0.277241,NaN
t4,0.358973,0.156552,0.093671,0.092135,0.178127,0.263108,NaN


Guardamos el ingreso promedio

In [74]:
for t in [1, 2, 3, 4]:
    df_temp = dict_t[t].loc[dict_t[t][f'ingr_t_t{t}'] >= 0].copy()
    
    # Calcula la suma total de los factores de expansión
    peso_total = df_temp['fexp'].sum()
    
    # Calcula el ingreso promedio ponderado
    media_ponderada = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total
    
    datos_final.loc[f't{t}', 'ingreso_promedio'] = media_ponderada

In [75]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.25763,0.107017,0.062802,0.077345,0.151542,0.226253,136.425138
t2,0.264598,0.099451,0.052628,0.082729,0.158865,0.232562,136.633803
t3,0.29742,0.122079,0.070364,0.097421,0.186945,0.277241,143.932641
t4,0.358973,0.156552,0.093671,0.092135,0.178127,0.263108,119.495271


### Inserta los cálculos en la base final

In [76]:
indices = pd.read_csv("indices.csv", encoding='latin-1')

In [77]:
ano = 2005
# Asegurar que el índice de datos_final coincide con trimestres 1..4
datos_final = datos_final.copy()
datos_final["trimestre"] = [1, 2, 3, 4]
datos_final["Año"] = ano

# Reemplazar en indices usando mask
for col in ["fgt0","fgt1","fgt2","a25","a50","a75","ingreso_promedio"]:
    indices.loc[indices["Año"].eq(ano), col] = datos_final[col].values

C:\Users\oscarj\AppData\Local\Temp\ipykernel_31344\3354817307.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.25763025351308944 0.26459837976052125 0.29742039844281093
 0.35897298876923034]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  indices.loc[indices["Año"].eq(ano), col] = datos_final[col].values
C:\Users\oscarj\AppData\Local\Temp\ipykernel_31344\3354817307.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.1070174395852279 0.09945116218976326 0.12207918358398304
 0.15655227775225164]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  indices.loc[indices["Año"].eq(ano), col] = datos_final[col].values
C:\Users\oscarj\AppData\Local\Temp\ipykernel_31344\3354817307.py:9: FutureWarning: Setting an item of incompatible dtype is

In [78]:
indices.to_csv('indices.csv', encoding='latin-1', index=None)